In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 22. Ensemble Learning: Gradient Boosting

## Algorithm Category
**Type**: Ensemble Learning - Classification/Regression  
**Complexity**: Medium-High  
**Use Case**: Sequential boosting using gradient descent to minimize loss

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the Gradient Boosting algorithm and how it differs from AdaBoost
- Implement Gradient Boosting for classification and regression
- Understand how gradient descent is used in boosting
- Visualize how the model improves over iterations
- Tune hyperparameters (n_estimators, learning_rate, max_depth)
- Apply Gradient Boosting to real-world problems

## Historical Context

Gradient Boosting was developed by Jerome Friedman in 2001:
- Friedman, J.H. (2001): "Greedy Function Approximation: A Gradient Boosting Machine"
- Generalization of boosting using gradient descent
- Foundation for XGBoost, LightGBM, and CatBoost

**Key Papers/References:**
- Friedman, J.H. (2001). "Greedy Function Approximation: A Gradient Boosting Machine"
- Friedman, J.H. (2002). "Stochastic Gradient Boosting"

## When to Use Gradient Boosting

Gradient Boosting is appropriate when:
- You need high accuracy
- Working with structured/tabular data
- Non-linear relationships in data
- You have time for hyperparameter tuning
- Moderate to large datasets
- Both classification and regression tasks

## Theory & Mechanics

### Mathematical Foundation

Gradient Boosting uses gradient descent to sequentially add weak learners that minimize the loss function.

**Initialization:**
$$F_0(x) = \arg\min_{\gamma} \sum_{i=1}^{N} L(y_i, \gamma)$$

**For each iteration t = 1, 2, ..., T:**

1. **Calculate residuals (negative gradients):**
   $$r_{it} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F(x)=F_{t-1}(x)}$$

2. **Fit weak learner to residuals:**
   $$h_t(x) = \arg\min_{h} \sum_{i=1}^{N} (r_{it} - h(x_i))^2$$

3. **Find optimal step size:**
   $$\gamma_t = \arg\min_{\gamma} \sum_{i=1}^{N} L(y_i, F_{t-1}(x_i) + \gamma h_t(x_i))$$

4. **Update model:**
   $$F_t(x) = F_{t-1}(x) + \gamma_t h_t(x)$$

**Final Prediction:**
$$\hat{y} = F_T(x) = F_0(x) + \sum_{t=1}^{T} \gamma_t h_t(x)$$

### How It Works

1. **Start**: Initialize with constant prediction (mean for regression, log-odds for classification)
2. **Calculate gradients**: Compute negative gradients (residuals) of loss function
3. **Fit weak learner**: Train decision tree to predict residuals
4. **Update model**: Add weak learner to ensemble with optimal step size
5. **Repeat**: Continue until convergence or max iterations

### Key Hyperparameters

- **n_estimators**: Number of boosting iterations
- **learning_rate**: Shrinks contribution of each tree (default: 0.1)
- **max_depth**: Maximum depth of weak learners (default: 3)
- **min_samples_split**: Minimum samples to split a node
- **subsample**: Fraction of samples to use for each tree (stochastic gradient boosting)
- **loss**: Loss function ('deviance' for classification, 'ls' for regression)

### Advantages

- High predictive accuracy
- Handles non-linear relationships well
- Flexible (can use different loss functions)
- Provides feature importance
- Works well with default parameters

### Limitations

- Sequential training (cannot parallelize easily)
- Sensitive to overfitting
- Requires careful hyperparameter tuning
- Can be slow for large datasets
- Memory intensive


## Implementation

Let's implement Gradient Boosting for both classification and regression.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    load_breast_cancer,  # Breast cancer classification dataset
    load_diabetes  # Diabetes regression dataset
)
from sklearn.ensemble import (
    GradientBoostingClassifier,  # Gradient Boosting for classification
    GradientBoostingRegressor  # Gradient Boosting for regression
)
from sklearn.model_selection import (
    train_test_split,  # Split data into train/test sets
    cross_val_score,  # Cross-validation scoring
    GridSearchCV  # Hyperparameter tuning
)
from sklearn.metrics import (
    accuracy_score,  # Calculate accuracy (for classification)
    mean_squared_error  # Calculate MSE (for regression)
)

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.supervised import (
    split_data,  # Split data into train/test sets
    evaluate_classifier,  # Evaluate classification models
    evaluate_regressor  # Evaluate regression models
)
from src.models.classification import (
    calculate_classification_metrics,  # Calculate precision, recall, F1, etc.
    plot_confusion_matrix  # Visualize confusion matrix
)
from src.models.ensemble import (
    extract_feature_importance,  # Extract feature importance from ensemble
    plot_feature_importance  # Visualize feature importance
)
from src.utils.benchmarking import benchmark_model_training  # Measure training time
from src.utils.validation import (
    validate_model_output,  # Check if predictions are valid
    check_cross_validation_stability  # Check CV stability
)

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# LOADING THE DATASET: Breast Cancer Classification
# ============================================

# load_breast_cancer() loads the Breast Cancer Wisconsin dataset from scikit-learn
# This is a binary classification problem: predict if tumor is malignant (1) or benign (0)
cancer = load_breast_cancer()  # Returns a Bunch object with data, target, feature_names

# X = Features (inputs): Medical measurements of breast tumors
# cancer.data contains feature values (569 samples × 30 features)
# We convert to DataFrame for easier manipulation
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
# Features include: mean radius, mean texture, mean perimeter, mean area, etc. (30 total)

# y = Target (output): Tumor type (what we want to predict)
# cancer.target contains class labels (0 = benign, 1 = malignant)
y = pd.Series(cancer.target, name='Target')
# 0 = benign (non-cancerous), 1 = malignant (cancerous)

print(f"Dataset Shape: {X.shape}")  # Output: (569, 30) - 569 patients, 30 features
print(f"Classes: {cancer.target_names.tolist()}")  # Output: ['malignant', 'benign']

# ============================================
# TRAIN/TEST SPLIT: Separating Data
# ============================================

# Note: Gradient Boosting doesn't require feature scaling (uses decision trees internally)
# Decision trees are scale-invariant (splits are based on comparisons, not distances)

# split_data() randomly splits data into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

# ============================================
# MODEL CREATION: Gradient Boosting Classifier
# ============================================

# Gradient Boosting builds models sequentially, each correcting errors of previous models
# Uses gradient descent to minimize loss function

# Create GradientBoostingClassifier
model = GradientBoostingClassifier(
    n_estimators=100,  # Number of boosting iterations (weak learners)
    #   - More estimators = better but slower
    #   - Typical values: 50-200
    learning_rate=0.1,  # Shrinks contribution of each tree
    #   - Lower = slower learning, more stable (default: 0.1)
    #   - Higher = faster learning, may overfit
    #   - Typical values: 0.01-0.3
    max_depth=3,  # Maximum depth of weak learners (decision trees)
    #   - Smaller = simpler trees, more regularization
    #   - Larger = more complex trees, may overfit
    #   - Default: 3 (good balance)
    random_state=42  # Reproducibility
)

# ============================================
# MODEL TRAINING: Sequential Gradient Descent
# ============================================

# .fit() trains the Gradient Boosting model
# The algorithm:
# 1. Initialize with constant prediction (mean for regression, log-odds for classification)
# 2. For each iteration:
#    a. Calculate negative gradients (residuals) of loss function
#    b. Train decision tree to predict these residuals
#    c. Find optimal step size (learning rate)
#    d. Add tree to ensemble with step size
# 3. Combine all trees for final prediction
model.fit(X_train, y_train)  # Train the model

print("\nGradient Boosting Classifier:")
print(f"Number of estimators: {model.n_estimators}")  # Number of trees (100)
print(f"Learning rate: {model.learning_rate}")  # Learning rate (0.1)
print(f"Max depth: {model.max_depth}")  # Maximum depth of trees (3)

# ============================================
# MAKING PREDICTIONS
# ============================================

# .predict() makes predictions by:
# 1. Each tree makes a prediction
# 2. Sum all predictions (weighted by learning rate)
# 3. Apply transformation (sigmoid for classification)
y_pred = model.predict(X_test)  # Class predictions (0 or 1)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)  # Compare predictions to true labels
print(f"\nTest Accuracy: {accuracy:.3f}")  # Display accuracy

# ============================================
# EVALUATING MODEL PERFORMANCE
# ============================================

# Evaluate with helper function
results = evaluate_classifier(model, X_test, y_test)
# Returns dictionary with accuracy and other metrics

# Calculate detailed classification metrics
metrics = calculate_classification_metrics(y_test.values, y_pred)
# Returns dictionary with: accuracy, precision, recall, F1

print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1_score']:.3f}")
# Precision: Of predicted positives, how many were actually positive
# Recall: Of actual positives, how many did we catch
# F1: Harmonic mean of precision and recall (balances both)


## Learning Curve

Let's visualize how the model improves over iterations.


In [ ]:
# ============================================
# LEARNING CURVE: Tracking Performance Over Iterations
# ============================================

# Gradient Boosting improves iteratively - each new tree corrects previous errors
# staged_predict() allows us to see predictions at each stage (after each tree)
# This shows how the model improves (or overfits) as we add more trees

# Use staged_predict to get predictions at each stage
train_scores = []  # Store training accuracy after each tree
test_scores = []  # Store test accuracy after each tree

# staged_predict() returns an iterator that yields predictions after each tree
# This lets us see how performance changes as we add trees
for train_pred, test_pred in zip(
    model.staged_predict(X_train),  # Predictions on training data (after each tree)
    model.staged_predict(X_test)  # Predictions on test data (after each tree)
):
    # train_pred: Predictions after adding one more tree (training data)
    # test_pred: Predictions after adding one more tree (test data)
    
    # Calculate accuracies at this stage
    train_scores.append(accuracy_score(y_train, train_pred))  # Training accuracy
    test_scores.append(accuracy_score(y_test, test_pred))  # Test accuracy

# ============================================
# VISUALIZING LEARNING CURVE
# ============================================

# Create line plot
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Plot training and test accuracy over iterations
plt.plot(range(1, len(train_scores) + 1), train_scores, 'o-', label='Training Accuracy', markersize=3)
plt.plot(range(1, len(test_scores) + 1), test_scores, 's-', label='Test Accuracy', markersize=3)
# range(1, len(...) + 1): X-axis (number of trees: 1, 2, 3, ..., 100)
# markersize=3: Small markers (many points)

# Label axes
plt.xlabel('Number of Estimators')  # X-axis: number of trees added
plt.ylabel('Accuracy')  # Y-axis: accuracy (0 to 1)
plt.title('Gradient Boosting: Learning Curve')  # Chart title
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# ============================================
# CHECKING FOR OVERFITTING
# ============================================

# Calculate overfitting gap (difference between training and test accuracy)
overfitting_gap = [t - s for t, s in zip(train_scores, test_scores)]
# t - s: Training accuracy - Test accuracy
# Large gap = model performs much better on training than test (overfitting)

# Check if overfitting is significant
if max(overfitting_gap) > 0.1:
    print(f"Warning: Potential overfitting detected (max gap: {max(overfitting_gap):.3f})")
    # Gap > 0.1 (10%) suggests overfitting
    # Solution: Reduce learning_rate, reduce max_depth, or use early stopping
else:
    print(f"Model shows good generalization (max gap: {max(overfitting_gap):.3f})")
    # Small gap = model generalizes well

# Interpretation:
# - Learning curve shows how model improves over iterations
# - Training accuracy usually increases (model fits training data better)
# - Test accuracy should also increase (model learns useful patterns)
# - If test accuracy plateaus or decreases while train increases = overfitting
# - Optimal number of trees: where test accuracy is highest


## Validation & Testing

Let's validate the model and check for overfitting.


In [ ]:
# ============================================
# VALIDATION 1: Cross-Validation
# ============================================

# Cross-validation splits data into k folds and tests on each fold
# More reliable than single train/test split
# cross_val_score() performs k-fold cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
# model: The trained Gradient Boosting model
# X, y: All data (will be split internally)
# cv=5: 5 folds (5 train/test splits)
# scoring='accuracy': Use accuracy as evaluation metric
# Returns: array of 5 accuracy scores (one per fold)

# Calculate statistics across folds
cv_mean = cv_scores.mean()  # Average accuracy across all folds
cv_std = cv_scores.std()  # Standard deviation (measure of variability)

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")  # Average ± variability

# ============================================
# STABILITY CHECK: Is Performance Consistent?
# ============================================

# Check if cross-validation results are stable (low variation)
# Unstable results suggest model is sensitive to data split
# check_cross_validation_stability() calculates coefficient of variation
stability = check_cross_validation_stability(cv_scores, threshold=0.1)
# threshold=0.1: Variation should be less than 10% of mean
# Returns dictionary with stability analysis

print(f"  Is Stable: {stability['is_stable']}")  # True if variation < threshold

# ============================================
# VALIDATION 2: Checking Model Output Validity
# ============================================

# validate_model_output() checks if predictions are valid
# task_type='classification' tells validator this is classification
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
# Returns dictionary with validation results

# ============================================
# ASSERTIONS: Automated Validation Checks
# ============================================

# Check 1: Predictions must be valid
assert validation_result['valid'], "Invalid predictions!"
# If predictions are invalid (wrong shape, wrong values, etc.), stop execution

# Check 2: Accuracy must be better than random guessing
# For binary classification, random = 0.5 (50%)
assert accuracy > 0.5, "Accuracy should be better than random!"
# If accuracy ≤ 0.5, model is no better than guessing

print("\n✓ Validation checks passed")  # All checks passed!


## Feature Importance

Let's extract and visualize feature importance.


In [ ]:
# Extract feature importance
feature_importance = extract_feature_importance(model, feature_names=X.columns.tolist())
print("Top 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize
plot_feature_importance(feature_importance, top_n=15, title="Gradient Boosting Feature Importance")


## Regression Example

Let's apply Gradient Boosting to regression.


In [ ]:
# ============================================
# REGRESSION EXAMPLE: Gradient Boosting for Continuous Values
# ============================================

# Gradient Boosting can also be used for regression (predicting continuous values)
# Instead of classification, it predicts a continuous number
# Uses same gradient descent approach but with regression loss function

# Load Diabetes dataset (regression problem)
diabetes = load_diabetes()  # Returns Bunch object
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)  # Features: medical measurements
y_reg = pd.Series(diabetes.target, name='Target')  # Target: disease progression (continuous number)

# ============================================
# TRAIN/TEST SPLIT
# ============================================

# Split data into training and test sets
X_reg_train, X_reg_test, y_reg_train, y_reg_test = split_data(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# ============================================
# TRAINING GRADIENT BOOSTING REGRESSOR
# ============================================

# Create GradientBoostingRegressor (for regression, not classification)
gb_reg = GradientBoostingRegressor(
    n_estimators=100,  # Number of boosting iterations
    learning_rate=0.1,  # Learning rate (same as classifier)
    max_depth=3,  # Maximum depth of trees (same as classifier)
    random_state=42  # Reproducibility
)
# GradientBoostingRegressor uses squared error loss (for regression)

# Train the model
gb_reg.fit(X_reg_train, y_reg_train)  # Train the Gradient Boosting regressor

# ============================================
# MAKING PREDICTIONS
# ============================================

# Make predictions on test set
y_reg_pred = gb_reg.predict(X_reg_test)  # Predictions: continuous values
# Each tree makes a prediction, then all predictions are summed

# ============================================
# EVALUATING REGRESSION PERFORMANCE
# ============================================

# Calculate RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
# mean_squared_error(): Calculates MSE (average squared error)
# np.sqrt(): Takes square root to get RMSE (in same units as target)
# Lower is better (0 = perfect predictions)

print("Gradient Boosting Regression:")
print(f"  Test RMSE: {rmse:.3f}")  # Display RMSE

# ============================================
# VISUALIZING PREDICTIONS
# ============================================

# Create scatter plot: predicted vs actual values
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Scatter plot: each point is one test sample
plt.scatter(y_reg_test, y_reg_pred, alpha=0.6)
# X-axis: actual values, Y-axis: predicted values
# alpha=0.6: Semi-transparent points (easier to see overlapping points)

# Plot diagonal line (perfect predictions)
plt.plot([y_reg_test.min(), y_reg_test.max()], 
         [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
# [y_reg_test.min(), y_reg_test.max()]: X-coordinates (actual min to max)
# [y_reg_test.min(), y_reg_test.max()]: Y-coordinates (same range)
# 'r--': Red dashed line
# lw=2: Line width
# This line represents perfect predictions (predicted = actual)

plt.xlabel('Actual Values')  # X-axis: true target values
plt.ylabel('Predicted Values')  # Y-axis: model predictions
plt.title('Gradient Boosting Regression: Predicted vs Actual')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# Interpretation:
# - Points close to red line = good predictions
# - Points far from red line = poor predictions
# - RMSE shows average distance from red line
# - For diabetes dataset, RMSE of ~50-60 is typical
# - Gradient Boosting regression works well for non-linear relationships


## Real-World Application

Let's tune hyperparameters and compare learning rates.


In [ ]:
# ============================================
# COMPARING LEARNING RATES: Finding Optimal Value
# ============================================

# Learning rate is crucial for Gradient Boosting
# Lower learning rate = slower learning, more stable (requires more estimators)
# Higher learning rate = faster learning, may overfit (requires fewer estimators)
# We'll test different values to find the optimal balance

# Test different learning rates
learning_rates = [0.01, 0.1, 0.5, 1.0]  # Range from very conservative to aggressive
lr_results = {}  # Store results for each learning rate

# Test each learning rate
for lr in learning_rates:
    # Create Gradient Boosting with this learning rate
    gb = GradientBoostingClassifier(
        n_estimators=100,  # Same number of estimators for fair comparison
        learning_rate=lr,  # Learning rate to test
        max_depth=3,  # Same max depth for all
        random_state=42  # Reproducibility
    )
    
    # Train the model
    gb.fit(X_train, y_train)  # Train on training data
    
    # Make predictions and calculate accuracy
    pred = gb.predict(X_test)  # Predictions on test set
    acc = accuracy_score(y_test, pred)  # Test accuracy
    
    # Store result
    lr_results[lr] = acc  # Store accuracy for this learning rate
    print(f"Learning rate {lr}: Test Accuracy = {acc:.3f}")

# ============================================
# FINDING OPTIMAL LEARNING RATE
# ============================================

# Find learning rate with highest test accuracy
best_lr = max(lr_results, key=lr_results.get)
# max(..., key=...): Finds learning rate with maximum accuracy
# lr_results.get: Extract accuracy value for comparison

print(f"\nBest learning rate: {best_lr} with accuracy: {lr_results[best_lr]:.3f}")

# Interpretation:
# - Learning rate = 0.1 is default (good balance)
# - Very low (0.01): Too slow, may underfit
# - High (0.5-1.0): Fast but may overfit
# - Optimal learning rate balances speed and stability

# ============================================
# HYPERPARAMETER TUNING: Comprehensive Search
# ============================================

# GridSearchCV tests all combinations of hyperparameters and finds the best
# This is more thorough than testing one parameter at a time

# param_grid defines the hyperparameters to search:
param_grid = {
    'n_estimators': [50, 100],  # Number of boosting iterations
    # More estimators = better but slower
    
    'learning_rate': [0.05, 0.1, 0.2],  # Learning rate
    # Lower = slower but more stable
    
    'max_depth': [3, 5]  # Maximum depth of trees
    # Smaller = simpler trees, more regularization
}
# Total combinations: 2 × 3 × 2 = 12 combinations to test

# ============================================
# GRID SEARCH: Testing All Combinations
# ============================================

# GridSearchCV performs cross-validation for each hyperparameter combination
# cv=3: 3-fold cross-validation (reduced from 5 for faster execution)
# scoring='accuracy': Use accuracy to evaluate each combination
# n_jobs=-1: Use all CPU cores (parallel processing, faster)
# verbose=1: Show progress (1 = some output)
grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),  # Base model
    param_grid,  # Hyperparameters to search
    cv=3,  # 3-fold cross-validation
    scoring='accuracy',  # Evaluation metric
    n_jobs=-1,  # Parallel processing
    verbose=1  # Show progress
)

# Train and evaluate all combinations
# This may take a while: 12 combinations × 3 folds = 36 models to train
grid_search.fit(X_train, y_train)

# ============================================
# DISPLAYING BEST RESULTS
# ============================================

print(f"\nBest Hyperparameters: {grid_search.best_params_}")  # Best combination found
print(f"Best CV Accuracy: {grid_search.best_score_:.3f}")  # Best cross-validation accuracy

# Interpretation:
# - Best hyperparameters are the ones that gave highest CV accuracy
# - CV accuracy is more reliable than single train/test split
# - Compare with single learning rate test: GridSearch is more comprehensive
# - Best hyperparameters may differ from default values


## Summary & Key Takeaways

### Key Concepts Learned

1. **Gradient Boosting Basics**
   - Sequential ensemble using gradient descent
   - Fits weak learners to residuals (negative gradients)
   - Minimizes loss function iteratively
   - More flexible than AdaBoost (works with any differentiable loss)

2. **Gradient Descent in Boosting**
   - Calculate gradients of loss function
   - Fit weak learner to negative gradients
   - Update model by adding weak learner
   - Repeat until convergence

3. **Key Hyperparameters**
   - **n_estimators**: Number of boosting iterations
   - **learning_rate**: Shrinks contribution (lower = more iterations needed)
   - **max_depth**: Controls complexity of weak learners
   - **subsample**: Fraction of samples per tree (stochastic boosting)

4. **Best Practices**
   - Use small learning rate (0.1) with more estimators
   - Monitor learning curve to detect overfitting
   - Use early stopping if available
   - Tune max_depth to control complexity

### When to Use Gradient Boosting

✅ **Good for:**
- High accuracy requirements
- Structured/tabular data
- Non-linear relationships
- Both classification and regression
- When you can tune hyperparameters

❌ **Not ideal for:**
- Very large datasets (use XGBoost or LightGBM)
- Real-time predictions (can be slow)
- When interpretability is crucial
- High-dimensional sparse data
- When parallelization is critical

### Next Steps

- Try **XGBoost** for optimized gradient boosting
- Explore **LightGBM** for faster training
- Consider **CatBoost** for categorical features
- Use **Early Stopping** to prevent overfitting
- Compare with **Random Forest** (bagging vs boosting)
